## Imports

In [ ]:
import json

with open('../../config.json', 'r') as f:
    config = json.load(f)

MAGNETSTEIN_PATH = config["magnetstein_path"]
SRC_PATH = config["src_path"]

import sys
sys.path.insert(0, MAGNETSTEIN_PATH)
sys.path.insert(1, SRC_PATH)
from utils import *

ADDITIONAL_ANALYSIS_RESULTS_PATHS = config["additional_analysis_results_paths_overlapping_simulations"]

## Scenario: overlapping + small unique peaks

### Theory

**1. Properties of Lorentz distribution**

Let us start from describing the basic properties of the Lorentz distribution to understand the further reasoning.

Lorentz distribution (i.e. Cauchy distribution) has two parameters: $x_0$ (location, real) and $\gamma$ (shape, positive).

The formula for PDF is as follows:

$ p(x) = \frac{1}{\pi \gamma \Big[1 + (\frac{x-x_{0}}{\gamma})^2\Big]} $

And for CDF:

$ f(x) = \frac{1}{\pi} \text{arctan}\Big(\frac{x-x_0}{\gamma}\Big) + \frac{1}{2}$

Lorentz distribution is symmetric around $x_0$ which we can see from the formula for PDF.

**2. Our task**

We would like to simulate two lorentzian peaks (i.e. distributions) with some overlap. Let's say that the first distribution has the location parameter $x_0$ and cumulative distribution function $f_0$ and that the second distribution has the location parameter $x_1$ and cumulative distribution function $f_2$. Let's assume that they share the same scale parameter $\gamma$ and that $x_1 > x_0$, i.e. the second distribution is located on the right hand side of the first one.

We would like to be able to change the amount of overlap. Without the loss of generality, we can fix $x_0$ and change only $x_1$. Now, we would like to know how to change $x_1$ to obtain a specific proportion of overlap $P$. So, basically, we would like to get a formula for a function calculating $x_1$ from $P$ somehow.

**3. Solution**

First, let us notice that the point of intersection of the two PDFs is exactly in the middle between $x_0$ and $x_1$, i.e. in $x^{*}=\frac{x_0+x_1}{2}$. (This is due to symmetry of the Lorentz distribution, but can be also explicitly calculated).

To calculate the area of overlap, we need to add two areas: the first one is the area to the left of $x^{*}$ and, at the same time, below $p_1$ and the second one is the area to the right of $x^{*}$ and, at the same time, below $p_0$. From symmetry it is enough to calculate one of them and multiply it by 2.

The area to the right of $x^{*}$ and, at the same time, below $p_1$ is exactly the value of CDF in point $x^*$, i.e. $f_1(\frac{x_0+x_1}{2})$. So the formula for proportion of overlap $P$ as a function of $x_1$ is:

$P = 2 \cdot f_1(\frac{x_0+x_1}{2})$

Now, we need to use the explicit formula for Lorentz distribution's CDF:

$P = \frac{2}{\pi} \text{arctan}\Big(\frac{x_0-x_1}{2\gamma}\Big) + 1$

Now we need to transform this equation to get the formula for $x_1$ as a function of $P$:

$\frac{\pi}{2}(P-1)=\text{arctan}\Big(\frac{x_0-x_1}{2\gamma}\Big)$

$x_1 = -2 \gamma \cdot \text{tan}\Big[\frac{\pi}{2}(P-1) \Big] + x_0$

From trigonometric properties:

$x_1 = 2 \gamma \cdot \text{tan}\Big[\frac{\pi}{2}(1-P) \Big] + x_0$

This is the function that we've been seeking. Note that it makes sense, because for $P=1$ the value is $x_0$ and for $P=0$ is undefined, becuase it is impossible for the two Lorentzian peaks on the real number line to have no overlap.

### Parameters

Set:

In [ ]:
first_peak_position = 2.5
scale = 0.03

small_peak_position = 1
scale = 0.03

chemical_shift_axis = np.arange(0, 5, 0.001)
ground_truth_proportions = [0.5, 0.5]

Adjustable:

In [ ]:
small_peak_proportion = 0.
shift_diff = 0.

overlap_proportion = np.arange(0.02, 1.01, 0.01)
shift = np.arange(0.001, 0.51, 0.02)

### Components

Set:

In [ ]:
#add_in_proportion

Adjustable:

In [ ]:
#second_peak_location

In [ ]:
#create_second_peak

### Mixture

In [ ]:
#create_mixture

### Estimation overlap + shift

In [ ]:
run_overlapping_simulations(overlap_proportion=overlap_proportion, shift=shift, chemical_shift_axis=chemical_shift_axis, 
                            first_peak_position=first_peak_position, small_peak_position=small_peak_position,
                            small_peak_proportion=small_peak_proportion, scale=scale, shift_diff=shift_diff, 
                            ground_truth_proportions=ground_truth_proportions, results_path=ADDITIONAL_ANALYSIS_RESULTS_PATHS)

### Visualisation: overlap + shift

In [ ]:
abs_error_df = pd.read_csv(ADDITIONAL_ANALYSIS_RESULTS_PATHS + 'estimation_error_small_peak_proportion_0.0_shift_diff_0.0.csv', header=0, index_col=0)

In [ ]:
visualise_overlap_and_shift(abs_error_df, ADDITIONAL_ANALYSIS_RESULTS_PATHS, small_peak_proportion, shift_diff)